In [4]:
import psycopg2
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import constants
from datetime import date, datetime, timedelta
load_dotenv()
pd.options.mode.chained_assignment = None

from dateutil.relativedelta import relativedelta
database_name = os.getenv('WAREHOUSE_NAME')
database_user = os.getenv('WAREHOUSE_USER')
database_password = os.getenv('WAREHOUSE_PASSWORD')
database_host = os.getenv('WAREHOUSE_HOST')
database_port = os.getenv('WAREHOUSE_PORT')

In [5]:
from extract import get_sales_same_month
from extract import get_overall_sales, get_overall_order, extract_order_target, extract_sales_target

In [6]:
from transform import filter_selected_day, col_to_date

In [7]:
df = get_sales_same_month('2024-11-01','2024-11-09','2024-10-01','2024-10-09')

In [8]:
df = col_to_date(df, ['sales_date','order_date'])

In [9]:
df['is_same_month'] = np.where(df['sales_date'].dt.month == df['order_date'].dt.month, 'ĐĐH trong tháng', 'ĐĐH cũ')

In [10]:
df['is_timber'] = np.where(df['factory_code']=='30673', 'Timber','Remain')

In [11]:
df['month'] = df['sales_date'].dt.month

In [12]:
df_no_timber = df[df['factory_code'] != '30673']

In [13]:
df_no_timber_grouped = df_no_timber.groupby('month').agg({'sales_quantity':'sum'})

In [14]:
df_no_timber_grouped['diff'] = df_no_timber_grouped['sales_quantity'].diff()
df_no_timber_grouped['pct_change'] = df_no_timber_grouped['sales_quantity'].pct_change()

In [15]:
df_no_timber_grouped

,sales_quantity,diff,pct_change
month,,,
10,173202.5,NaN,NaN
11,147499.0,-25703.5,-0.148401


In [16]:
df_grouped = df.groupby('month').agg({'sales_quantity':'sum'})

In [17]:
df_grouped['diff'] = df_grouped['sales_quantity'].diff()
df_grouped['pct_change'] = df_grouped['sales_quantity'].pct_change()

In [18]:
df_grouped

,sales_quantity,diff,pct_change
month,,,
10,227304.5,NaN,NaN
11,195746.0,-31558.5,-0.138838


In [19]:
from extract import get_order_same_month
df_order = get_order_same_month('2024-11-01','2024-11-09','2024-10-01','2024-10-09')

In [20]:
df_order['order_date'] = pd.to_datetime(df_order['order_date'])
df_order['month'] = df_order['order_date'].dt.month

In [21]:
df_order_no_timber = df_order[df_order['factory_code'] != '30673']

In [22]:
df_order_no_timber_grouped = df_order_no_timber.groupby('month').agg({'order_quantity':'sum'})

In [23]:
df_order_no_timber_grouped['diff'] = df_order_no_timber_grouped['order_quantity'].diff()
df_order_no_timber_grouped['pct_change'] = df_order_no_timber_grouped['order_quantity'].pct_change()

In [24]:
df_order_grouped = df_order.groupby('month').agg({'order_quantity':'sum'}).reset_index()

In [25]:
df_order_grouped['diff'] = df_order_grouped['order_quantity'].diff()
df_order_grouped['pct_change'] = df_order_grouped['order_quantity'].pct_change()

In [26]:
df_order_grouped

,month,order_quantity,diff,pct_change
0,10,269512.5,NaN,NaN
1,11,116864.0,-152648.5,-0.566387


In [27]:
df_order_no_timber_grouped

,order_quantity,diff,pct_change
month,,,
10,162777.5,NaN,NaN
11,116864.0,-45913.5,-0.282063


In [28]:
df_same_month_grouped = df.groupby(['month','is_same_month']).agg({'sales_quantity':'sum'}).reset_index()

In [29]:
df_same_month_grouped

,month,is_same_month,sales_quantity
0,10,ĐĐH cũ,125047.0
1,10,ĐĐH trong tháng,102257.5
2,11,ĐĐH cũ,123079.0
3,11,ĐĐH trong tháng,72667.0


In [30]:
df_same_month_grouped = df_same_month_grouped.pivot(index='month',
                                                    columns='is_same_month',
                                                    values='sales_quantity').reset_index()

In [31]:
df_same_month_grouped

is_same_month,month,ĐĐH cũ,ĐĐH trong tháng
0,10,125047.0,102257.5
1,11,123079.0,72667.0


In [32]:
df_all = df_same_month_grouped[['month','ĐĐH cũ','ĐĐH trong tháng']].merge(df_order_grouped[['month','order_quantity']],
                                                                           on='month', how='left')

In [33]:
df_all['sales_quantity'] = df_all['ĐĐH cũ'] + df_all['ĐĐH trong tháng']

In [ ]:
df_all

,month,ĐĐH cũ,ĐĐH trong tháng,order_quantity,sales_quantity
0,10,125047.0,102257.5,269512.5,227304.5
1,11,123079.0,72667.0,116864.0,195746.0


In [35]:
df = df_all.copy()

In [36]:
import plotly.graph_objects as go
import pandas as pd


# Create the figure
fig = go.Figure()

# Add stacked bar traces
fig.add_trace(
    go.Bar(
        x=df["month"],
        y=df["ĐĐH cũ"],
        name="ĐĐH cũ",
        marker_color="#FFCC99",
    )
)
fig.add_trace(
    go.Bar(
        x=df["month"],
        y=df["ĐĐH trong tháng"],
        name="ĐĐH trong tháng",
        marker_color="#FFB9B9",
    )
)

# Add line traces
fig.add_trace(
    go.Scatter(
        x=df["month"],
        y=df["order_quantity"],
        name="Order Quantity",
        mode="lines+markers",
        line=dict(color="red", width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=df["month"],
        y=df["sales_quantity"],
        name="Sales Quantity",
        mode="lines+markers",
        line=dict(color="green", width=2),
    )
)

# Update layout for better readability
fig.update_layout(
    title="Stacked Bar Chart with Line Chart",
    xaxis=dict(title="Month"),
    yaxis=dict(title="Quantity"),
    barmode="stack",
    legend=dict(title="Legend"),
    template="plotly_white",
)

# Show the figure
fig.show()


In [37]:
start_date = '2024-11-01'
end_date = '2024-11-09'
start_date_target = '2024-10-01'
end_date_target = '2024-10-09'

In [38]:
start_date = datetime.strptime(start_date, '%Y-%m-%d')
end_date = datetime.strptime(end_date, '%Y-%m-%d')
start_date_target = datetime.strptime(start_date_target, '%Y-%m-%d')
end_date_target = datetime.strptime(end_date_target, '%Y-%m-%d')

In [44]:
start_date.day

1

In [39]:
df_sales = get_sales_same_month(start_date, end_date, start_date_target, end_date_target)
df_sales = col_to_date(df_sales, ['sales_date','order_date'])
df_sales['is_same_month'] = np.where(df_sales['sales_date'].dt.month == df_sales['order_date'].dt.month, 'ĐĐH trong tháng', 'ĐĐH cũ')
df_sales['month'] = df_sales['sales_date'].dt.month


In [ ]:

df_sales_no_timber = df_sales[df_sales['factory_code'] != '30673']
df_sales_no_timber_grouped = df_sales_no_timber.groupby('month').agg({'sales_quantity':'sum'})
df_sales_no_timber_grouped['diff'] = df_sales_no_timber_grouped['sales_quantity'].diff()
df_sales_no_timber_grouped['pct_change'] = df_sales_no_timber_grouped['sales_quantity'].pct_change()


In [60]:
df_sales_no_timber_grouped.iloc[-1,-1]

np.float64(-0.14840143762359093)

In [ ]:

df_sales_grouped = df_sales.groupby('month').agg({'sales_quantity':'sum'})
df_sales_grouped['diff'] = df_sales_grouped['sales_quantity'].diff()
df_sales_grouped['pct_change'] = df_sales_grouped['sales_quantity'].pct_change()


In [62]:
df_sales_grouped.iloc[-1,-1]

np.float64(-0.1388379904489352)

In [ ]:

df_order = get_order_same_month(start_date, end_date, start_date_target, end_date_target)
df_order = col_to_date(df_order, ['order_date'])
df_order['month'] = df_order['order_date'].dt.month

df_order_no_timber = df_order[df_order['factory_code'] != '30673']
df_order_no_timber_grouped = df_order_no_timber.groupby('month').agg({'order_quantity':'sum'})
df_order_no_timber_grouped['diff'] = df_order_no_timber_grouped['order_quantity'].diff()
df_order_no_timber_grouped['pct_change'] = df_order_no_timber_grouped['order_quantity'].pct_change()


In [57]:
df_order_no_timber_grouped.iloc[-1,-1]

np.float64(-0.2820629386739568)

In [ ]:

df_order_grouped = df_order.groupby('month').agg({'order_quantity':'sum'}).reset_index()
df_order_grouped['diff'] = df_order_grouped['order_quantity'].diff()
df_order_grouped['pct_change'] = df_order_grouped['order_quantity'].pct_change()


In [58]:
df_order_grouped

,month,order_quantity,diff,pct_change
0,10,269512.5,NaN,NaN
1,11,116864.0,-152648.5,-0.566387


In [ ]:

df_same_month_grouped = df_sales.groupby(['month','is_same_month']).agg({'sales_quantity':'sum'}).reset_index()
df_same_month_grouped = df_same_month_grouped.pivot(index='month',
                                                columns='is_same_month',
                                                values='sales_quantity').reset_index()
df_all = df_same_month_grouped[['month','ĐĐH cũ','ĐĐH trong tháng']].merge(df_order_grouped[['month','order_quantity']],
                                                                        on='month', how='left')
df_all['sales_quantity'] = df_all['ĐĐH cũ'] + df_all['ĐĐH trong tháng']

In [41]:
df_all

,month,ĐĐH cũ,ĐĐH trong tháng,order_quantity,sales_quantity
0,10,125047.0,102257.5,269512.5,227304.5
1,11,123079.0,72667.0,116864.0,195746.0
